# Optimizacion con Optuna + factorial dirigido

Busqueda eficiente de la mejor configuracion de forecasting para WPUSI01102B.

**Metodologia (split en 3 vias):**
- **Train** (<=2019): ajuste de modelos
- **Validacion** (2020-2021): Optuna optimiza aqui (incluye el shock COVID)
- **Test** (2022+): evaluacion final, INTACTO durante la busqueda

**Espacio de busqueda:** modelo x ventana grande {24,36,48} x transformacion x
16 subconjuntos de exogenas (2^4) x TDA {si,no} x estrategia anti-COVID.

Al final, un **factorial dirigido** sobre los mejores modelos da la tabla
comparativa limpia, evaluada en test.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import warnings; warnings.filterwarnings("ignore")
import logging
for _l in ["tensorflow","prophet","cmdstanpy","optuna"]:
    logging.getLogger(_l).setLevel(logging.ERROR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from berry_price_tda.pipelines.optimize import (
    run_optuna_search, evaluate_on_test, directed_factorial_on_test,
)
DATA = "../../data/interim/berry_features.csv"


## 1. Busqueda con Optuna (sobre validacion)

In [ ]:
study, best_config, df_trials = run_optuna_search(
    data_path=DATA,
    n_trials=200,        # sube a 300-500 para busqueda mas exhaustiva
    metric="mae",
    allow_tda=True,
    seed=42,
    verbose=True,
)
print("\nMejor configuracion:", best_config.label())

## 2. Historia de la optimizacion

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
vals = df_trials["value"].values
best_so_far = pd.Series(vals).cummin()
ax.plot(vals, 'o', alpha=0.4, label="MAE por trial", color="#888780")
ax.plot(best_so_far, '-', lw=2, label="Mejor hasta el momento", color="#534AB7")
ax.set_xlabel("Trial"); ax.set_ylabel("MAE (validacion)")
ax.set_title("Convergencia de Optuna"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Importancia de hiperparametros (que factores importan mas)
try:
    import optuna
    imp = optuna.importance.get_param_importances(study)
    imp_s = pd.Series(imp).sort_values()
    fig, ax = plt.subplots(figsize=(9,5))
    imp_s.plot(kind="barh", ax=ax, color="#1D9E75")
    ax.set_title("Importancia de cada factor (Optuna)")
    ax.set_xlabel("Importancia relativa"); plt.tight_layout(); plt.show()
except Exception as e:
    print("No se pudo calcular importancia:", e)

## 3. Evaluacion final en TEST (config ganadora)

In [ ]:
m = evaluate_on_test(DATA, best_config)
print("Config:", best_config.label())
print(f"MAE  = {m['mae']:.3f}")
print(f"RMSE = {m['rmse']:.3f}")
print(f"MAPE = {m['mape']:.2f}%")
print(f"R2   = {m['r2']:.3f}")

## 4. Factorial dirigido sobre TEST

Compara los mejores modelos con/sin TDA y con/sin las exogenas de la mejor
config, todo evaluado sobre el test. Esta es la tabla comparativa para el reporte.

In [ ]:
factorial = directed_factorial_on_test(DATA, best_config, verbose=True)
factorial.head(15)

In [ ]:
# Efecto del TDA en el factorial dirigido
piv = factorial.pivot_table(index="modelo", columns="use_tda", values="mae")
if piv.shape[1] == 2:
    piv.columns = ["sin_TDA", "con_TDA"]
    piv["delta"] = piv["con_TDA"] - piv["sin_TDA"]
    print("Efecto del TDA sobre MAE (positivo = empeora):")
    display(piv.round(3))

## 5. Guardar resultados

In [ ]:
df_trials.to_csv("../../data/processed/optuna_trials.csv", index=False)
factorial.to_csv("../../data/processed/factorial_dirigido_test.csv", index=False)
print("Resultados guardados en data/processed/")